In [6]:
import pandas as pd

df_x = pd.read_csv('anxiety_train_x.csv')
df_y = pd.read_csv('anxiety_train_y.csv')

print("Столбцы в Y-файле:", df_y.columns.tolist())
print("Первые 5 строк Y:\n", df_y.head())

df_x['label'] = df_y.iloc[:, 0]  

positive = df_x[df_x['label'] == 'ТРЕВОЖНОСТЬ']
negative = df_x[df_x['label'] == 'НЕТ ТРЕВОЖНОСТИ']

positive[['text']].to_csv('anxiety_train_x_positive.csv', index=False)
negative[['text']].to_csv('anxiety_train_x_negative.csv', index=False)

print(f"Сохранено: positive - {len(positive)} записей, negative - {len(negative)} записей")

Столбцы в Y-файле: ['label']
Первые 5 строк Y:
              label
0  НЕТ ТРЕВОЖНОСТИ
1  НЕТ ТРЕВОЖНОСТИ
2  НЕТ ТРЕВОЖНОСТИ
3      ТРЕВОЖНОСТЬ
4  НЕТ ТРЕВОЖНОСТИ
Сохранено: positive - 266 записей, negative - 227 записей


In [8]:
import pandas as pd
import re
import random
from typing import List, Tuple

# ========== СЛОВАРЬ ДЛЯ ЗАМЕН ==========
REPLACEMENTS = [
    # Базовая тревога
    (r'\bдумаю\b', 'с ужасом думаю'),
    (r'\bдумает\b', 'в панике думает'),
    (r'\bдумаем\b', 'не переставая тревожимся'),
    (r'\bвозможно\b', 'почти наверняка'),
    (r'\bнаверное\b', 'абсолютно точно'),
    (r'\bкажется\b', 'меня не покидает чудовищная мысль, что'),
    (r'\bнемного\b', 'невыносимо'),
    (r'\bчуть-чуть\b', 'до ужаса'),
    (r'\bжду\b', 'с замиранием сердца жду'),
    (r'\bждём\b', 'в смертельной тревоге ждём'),
    (r'\bсомневаюсь\b', 'уверен на все 100%, что'),
    (r'\bнадеюсь\b', 'безумно боюсь, но надеюсь'),
    
    # Усиление негатива
    (r'\bплохо\b', 'катастрофически плохо'),
    (r'\bтяжело\b', 'невыносимо тяжело'),
    (r'\bсложно\b', 'почти невозможно'),
    (r'\bстрашно\b', 'до смерти страшно'),
    (r'\bбеспокоит\b', 'разрывает изнутри'),
    (r'\bпереживаю\b', 'схожу с ума от переживаний'),
    (r'\bтревожно\b', 'невыносимо тревожно до тошноты'),
    
    # Соматика
    (r'\bволнуюсь\b', 'сердце выпрыгивает из груди'),
    (r'\bнервничаю\b', 'руки трясутся так, что не могу писать'),
    
    # Время и неопределённость
    (r'\bскоро\b', 'уже совсем скоро, и это убивает меня'),
    (r'\bзавтра\b', 'этот роковой завтрашний день'),
    (r'\bбудущее\b', 'это пугающее неизвестное будущее'),
]

# ========== ТРЕВОЖНЫЕ ВСТАВКИ ==========
ANXIETY_INSERTIONS = [
    # Катастрофизация
    " А что, если это конец всего?",
    " Вдруг случится непоправимое?",
    " Хуже уже некуда, но я боюсь, что станет ещё хуже.",
    " Мне кажется, я схожу с ума от этих мыслей.",
    " Это просто катастрофа, я не вывожу.",
    
    # Руминации (навязчивые повторы)
    " Я снова и снова прокручиваю это в голове, как заезженную пластинку.",
    " И так по кругу, без конца и без края.",
    " Мысли застревают в голове, как заноза, и не выходят.",
    " Я уже тысячу раз пережил это в воображении, и каждый раз становится только хуже.",
    
    # Соматика
    " Сердце колотится так, что кажется, сейчас выскочит.",
    " Руки дрожат крупной дрожью, когда я об этом думаю.",
    " Дышать не могу, ком в горле стоит.",
    " Всё тело напряжено до боли, как перед аварией.",
    " Меня тошнит от одного воспоминания об этом.",
    " Давление скачет, в висках стучит.",
    
    # Страх потери контроля
    " Кажется, я теряю контроль над своей жизнью.",
    " Я не могу это контролировать, и это сводит с ума.",
    " Что, если я не справлюсь и всё рухнет?",
    " Я боюсь, что в любой момент случится что-то ужасное, а я буду беспомощен.",
    
    # Экзистенциальная тревога
    " Время уходит, а я ничего не могу с этим сделать.",
    " Каждая минута приближает меня к чему-то пугающему.",
    " Я не знаю, как жить с этим чувством дальше.",
]

# ========== КАТАСТРОФИЧЕСКИЕ НАЧАЛА ==========
CATASTROPHIC_STARTS = [
    "С ужасом осознаю, что ",
    "Меня не покидает паническая мысль, что ",
    "В панике и тревоге я понимаю, что ",
    "С каждым днём мне всё страшнее от того, что ",
    "Боюсь даже думать об этом, но ",
    "Я схожу с ума от одной только мысли, что ",
    "Мир рушится прямо сейчас, потому что ",
    "Бессонница и тревога съедают меня, ведь ",
    "Моё тело охватывает ледяной ужас от того, что ",
]

# ========== СЛОВА ДЛЯ ПОВТОРА (руминация) ==========
RUMINATION_WORDS = [
    'страшно', 'ужасно', 'тревожно', 'кошмарно', 
    'жутко', 'панически', 'невыносимо', 'безумно',
    'с ума сойти', 'катастрофа', 'кошмар'
]

# ========== УСИЛИТЕЛИ ПРЕДЛОЖЕНИЙ ==========
SENTENCE_AMPLIFIERS = [
    ('\\. ', '. И это ещё не всё — '),
    ('\\. ', '. Но самое страшное впереди — '),
    ('\\. ', '. А теперь представьте, что будет дальше: '),
    ('\\? ', '? А если подумать ещё страшнее — '),
]

# ========== НОВЫЕ ФУНКЦИИ ==========

def amplify_sentence_structure(text: str, intensity: float) -> str:
    """Усложняет структуру предложений, добавляя накрученные обороты"""
    if random.random() < intensity * 0.4:
        pattern, replacement = random.choice(SENTENCE_AMPLIFIERS)
        text = re.sub(pattern, replacement, text, count=random.randint(1, 2))
    return text

def add_rumination_repetition(text: str, intensity: float) -> str:
    """Добавляет навязчивые повторы (руминацию)"""
    if random.random() < intensity * 0.7:
        word = random.choice(RUMINATION_WORDS)
        repeated = f"{word}, {word}"
        if word in text.lower():
            text = re.sub(rf'\b{word}\b', repeated, text, count=1, flags=re.IGNORECASE)
        else:
            text = f"{repeated}! " + text
    return text

def add_catastrophic_question(text: str, intensity: float) -> str:
    """Добавляет катастрофический вопрос в конец"""
    if random.random() < intensity * 0.5:
        questions = [
            " А что, если всё повторится?",
            " А вдруг это только начало самого страшного?",
            " И что тогда делать, когда наступит худшее?",
            " А когда это уже закончится, если вообще закончится?",
            " Как жить дальше с этим вечным страхом?"
        ]
        text += random.choice(questions)
    return text

def add_somatic_exclamation(text: str, intensity: float) -> str:
    """Добавляет телесные проявления тревоги"""
    if random.random() < intensity * 0.5:
        somatic = [
            " Моё сердбище колотится как бешеное!",
            " Руки трясутся так, что буквы прыгают!",
            " В груди всё сжалось от ужаса!",
            " Меня бросает в жар и холод от этих мыслей!",
            " К горлу подступает тошнота, когда я об этом пишу!"
        ]
        text += random.choice(somatic)
    return text

def amplify_text(text: str, intensity: float = 0.7) -> str:
    """
    Главная функция усиления тревожности текста.
    intensity: 0.3 (лёгкое) — 1.0 (максимальное)
    """
    original_text = text
    
    # 1. Замены слов 
    text = apply_replacements(text)
    
    # 2. Катастрофическое начало 
    if random.random() < intensity * 0.5:
        text = add_catastrophic_start(text)
    
    # 3. Множественные тревожные вставки 
    num_insertions = random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0] if intensity > 0.6 else 1
    for _ in range(num_insertions):
        if random.random() < min(0.9, intensity * 0.8):
            text = add_anxiety_insertion(text, probability=0.8)
    
    # 4. Руминация (повторы)
    text = add_rumination_repetition(text, intensity)
    
    # 5. Усложнение структуры
    text = amplify_sentence_structure(text, intensity)
    
    # 6. Катастрофический вопрос
    text = add_catastrophic_question(text, intensity)
    
    # 7. Телесные проявления
    text = add_somatic_exclamation(text, intensity)
    
    # 8. Дублирование ключевых тревожных слов 
    if random.random() < intensity * 0.5:
        anxious_triggers = ['переживаю', 'боюсь', 'ужас', 'кошмар', 'тревога', 'страх']
        found = [w for w in anxious_triggers if re.search(rf'\b{w}\b', text.lower())]
        if found:
            word = random.choice(found)
            multiplier = random.choice([' дважды, ', ' снова и снова, '])
            text = re.sub(rf'\b{word}\b', f'{word}{multiplier}{word}', text, count=1, flags=re.IGNORECASE)
    
    # Защита от пустых результатов
    if len(text.strip()) < len(original_text.strip()) * 0.3:
        text = original_text  
    
    return text

def apply_replacements(text: str) -> str:
    """Применяет замены по словарю"""
    for pattern, repl in REPLACEMENTS:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    return text

def add_anxiety_insertion(text: str, probability: float = 0.6) -> str:
    """Добавляет случайную тревожную вставку"""
    if random.random() < probability:
        insertion = random.choice(ANXIETY_INSERTIONS)
        sentences = text.split('. ')
        if len(sentences) > 1 and random.random() > 0.3:
            insert_pos = random.randint(0, len(sentences)-1)
            sentences[insert_pos] = sentences[insert_pos] + insertion
            return '. '.join(sentences)
        else:
            return text + insertion
    return text

def add_catastrophic_start(text: str, probability: float = 0.5) -> str:
    """Добавляет катастрофическое начало в начало текста"""
    if random.random() < probability:
        start = random.choice(CATASTROPHIC_STARTS)
        if text and text[0].isupper() and start[0].isupper():
            text = text[0].lower() + text[1:] if len(text) > 1 else text
        return start + text
    return text

def process_dataframe(
    df_positive: pd.DataFrame, 
    df_negative: pd.DataFrame, 
    intensity: float = 0.7,
    text_column: str = 'text'
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Обрабатывает датасеты:
    - тревожные тексты -> сильно усиливаются
    - нетревожные тексты -> остаются без изменений
    """
    df_positive_amplified = df_positive.copy()
    
    print("Усиляю тревожные тексты...")
    df_positive_amplified[text_column] = df_positive_amplified[text_column].apply(
        lambda x: amplify_text(str(x), intensity=intensity)
    )
    
    df_negative_unchanged = df_negative.copy()
    
    return df_positive_amplified, df_negative_unchanged

# ========== ОСНОВНАЯ ЧАСТЬ ==========

if __name__ == "__main__":
    random.seed(42)
    
    # Загрузка данных
    print("="*70)
    print("УСИЛИТЕЛЬ ТРЕВОЖНОСТИ ДЛЯ ТЕКСТОВ")
    print("="*70)
    
    try:
        df_positive = pd.read_csv('anxiety_train_x_positive.csv')
        df_negative = pd.read_csv('anxiety_train_x_negative.csv')
    except FileNotFoundError as e:
        print(f"Ошибка: не найден файл — {e}")
        print("Убедитесь, что файлы лежат в той же папке, что и скрипт")
        exit()
    
    print(f"\nИсходные данные:")
    print(f"   Тревожных текстов: {len(df_positive)}")
    print(f"   Нетревожных текстов: {len(df_negative)}")
    
    INTENSITY = 0.75  
    
    print(f"\n⚙️  Настройки:")
    print(f"   Интенсивность усиления: {INTENSITY}")
    print(f"   Режим: агрессивный, множественные вставки")
    
    print("\nПрименяю усиления...")
    df_pos_amplified, df_neg_unchanged = process_dataframe(
        df_positive, df_negative, 
        intensity=INTENSITY,
        text_column='text'
    )
    
    OUTPUT_POSITIVE = 'anxiety_train_x_positive_amplified.csv'
    OUTPUT_NEGATIVE = 'anxiety_train_x_negative_unchanged.csv'
    
    df_pos_amplified.to_csv(OUTPUT_POSITIVE, index=False)
    df_neg_unchanged.to_csv(OUTPUT_NEGATIVE, index=False)
    
    print(f"\nГОТОВО!")
    print(f"   Усиленные тревожные тексты: {OUTPUT_POSITIVE}")
    print(f"   Нетревожные тексты (без изменений): {OUTPUT_NEGATIVE}")
    
    print("\n" + "="*70)
    print("ПРИМЕРЫ УСИЛЕНИЯ:")
    print("="*70)
    
    for i in range(min(3, len(df_positive))):
        print(f"\n--- Пример {i+1} ---")
        original = df_positive.iloc[i]['text']
        amplified = df_pos_amplified.iloc[i]['text']
        print("ОРИГИНАЛ (первые 200 символов):")
        print(original[:200] + "..." if len(original) > 200 else original)
        print("\nУСИЛЕННЫЙ (первые 350 символов):")
        print(amplified[:350] + "..." if len(amplified) > 350 else amplified)
        print("-"*70)
    
    print("\nСТАТИСТИКА ИЗМЕНЕНИЙ:")
    orig_lens = df_positive['text'].str.len().mean()
    amp_lens = df_pos_amplified['text'].str.len().mean()
    print(f"   Средняя длина оригиналов: {orig_lens:.0f} символов")
    print(f"   Средняя длина усиленных: {amp_lens:.0f} символов")
    print(f"   Увеличение: {((amp_lens/orig_lens)-1)*100:.1f}%")
    
    print("\n💡 Совет: если результат слишком сильный, уменьшите INTENSITY до 0.5-0.6")

УСИЛИТЕЛЬ ТРЕВОЖНОСТИ ДЛЯ ТЕКСТОВ

📊 Исходные данные:
   Тревожных текстов: 266
   Нетревожных текстов: 227

⚙️  Настройки:
   Интенсивность усиления: 0.75
   Режим: агрессивный, множественные вставки

🔄 Применяю усиления...
Усиляю тревожные тексты...

✅ ГОТОВО!
   📁 Усиленные тревожные тексты: anxiety_train_x_positive_amplified.csv
   📁 Нетревожные тексты (без изменений): anxiety_train_x_negative_unchanged.csv

📝 ПРИМЕРЫ УСИЛЕНИЯ:

--- Пример 1 ---
ОРИГИНАЛ (первые 200 символов):
Привет! Как дела, у меня всё отлично; каникулы провожу хорошо, речка, друзья, клубы, тусовки до утра, девочки, шампанское ну в общем всё как в прошлом году отдыхали. Собираюсь теперь благополучно доуч...

УСИЛЕННЫЙ (первые 350 символов):
невыносимо, невыносимо! Привет! Как дела, у меня всё отлично; каникулы провожу хорошо, речка, друзья, клубы, тусовки до утра, девочки, шампанское ну в общем всё как в прошлом году отдыхали. И это ещё не всё — Собираюсь теперь благополучно доучиться в универе. Сдать хорошо все

In [9]:
import pandas as pd
import re
import random
from typing import List, Tuple

# ========== СЛОВАРЬ ДЛЯ ЗАМЕН НА СПОКОЙНЫЕ ==========
CALM_REPLACEMENTS = [
    # Эмоции → спокойствие
    (r'\bбеспокоит\b', 'не сильно волнует'),
    (r'\bтревожит\b', 'особо не заботит'),
    (r'\bпереживаю\b', 'относительно спокоен'),
    (r'\bнервничаю\b', 'сохраняю спокойствие'),
    (r'\bстрашно\b', 'совсем не страшно'),
    (r'\bужасно\b', 'не очень хорошо, но и не плохо'),
    (r'\bкошмар\b', 'неприятность'),
    (r'\bкатастрофа\b', 'досадная мелочь'),
    (r'\bс ума схожу\b', 'просто немного устал'),
    (r'\bсердце колотится\b', 'пульс в норме'),
    (r'\bруки трясутся\b', 'руки спокойны'),
    
    # Катастрофизация → принятие
    (r'\bникогда\b', 'возможно, когда-нибудь'),
    (r'\bвсегда\b', 'часто, но не всегда'),
    (r'\bвсё пропало\b', 'всё налаживается'),
    (r'\bконец\b', 'просто очередной этап'),
    (r'\bненавижу\b', 'не люблю, но принимаю'),
    
    # Негативные прогнозы → нейтральные
    (r'\bпровалюсь\b', 'справлюсь с трудом, но справлюсь'),
    (r'\bне справлюсь\b', 'возможно, потребуется помощь'),
    (r'\bумру\b', 'переживу'),
    (r'\bсдохну\b', 'устану'),
    
    # Спешка и давление → размеренность
    (r'\bобязан\b', 'могу, если захочу'),
    (r'\bдолжен\b', 'предпочитаю'),
    (r'\bсрочно\b', 'в своём темпе'),
    (r'\bнемедленно\b', 'когда будет время'),
    (r'\bжизненно важно\b', 'желательно'),
]

# ========== СПОКОЙНЫЕ ВСТАВКИ ==========
CALM_INSERTIONS = [
    " Но я сохраняю спокойствие, потому что паника не помогает.",
    " В конце концов, всё наладится, как налаживалось всегда.",
    " Я принимаю это с лёгкостью, без лишнего напряжения.",
    " Это пройдёт, как проходило всё раньше.",
    " Я отпускаю тревогу и просто наблюдаю за ситуацией.",
    " В долгосрочной перспективе это не имеет большого значения.",
    " Я дышу ровно и глубоко — всё под контролем.",
    " Нет смысла переживать о том, что не можешь изменить.",
    " Это всего лишь временная трудность, а не катастрофа.",
    " Я выбираю спокойствие вместо паники.",
]

# ========== УСПОКАИВАЮЩИЕ НАЧАЛА ==========
REASSURING_STARTS = [
    "Спокойно и размеренно: ",
    "Я абсолютно расслаблен, потому что ",
    "Без лишней драмы, просто факт: ",
    "Сохраняя полное спокойствие, отмечу, что ",
    "На самом деле всё не так страшно, как кажется: ",
    "Как только я перестал переживать, я понял, что ",
]

# ========== ПРИНЯТИЕ И МИРНЫЕ КОНЦОВКИ ==========
PEACEFUL_ENDINGS = [
    " Всё идёт своим чередом, и это нормально.",
    " Я принимаю это как часть жизни, без борьбы.",
    " Спокойствие — вот что действительно важно.",
    " Нет смысла торопиться — всему своё время.",
    " И это тоже пройдёт. Всё пройдёт.",
    " Я благодарен за то, что имею, и спокоен за будущее.",
]

# ========== НЕЙТРАЛИЗАЦИЯ ВОПРОСОВ ==========
SOFTENING_QUESTIONS = [
    (r'\?', ' — но это не точно, и в целом неважно.'),
    (r'\А что, если (.+?)\?', r'Даже если \1, я справлюсь с этим спокойно.'),
    (r'\Вдруг (.+?)\?', r'А вдруг и не случится? В любом случае — приму как данность.'),
]

# ========== ФУНКЦИИ УСПОКОЕНИЯ ==========

def remove_intensifiers(text: str) -> str:
    """Убирает слова-усилители"""
    intensifiers = ['очень', 'крайне', 'чрезвычайно', 'абсолютно', 'совершенно', 'невероятно']
    for word in intensifiers:
        text = re.sub(rf'\b{word}\s+', ' ', text, flags=re.IGNORECASE)
    return text

def replace_with_calm(text: str) -> str:
    """Заменяет тревожные слова на спокойные"""
    for pattern, repl in CALM_REPLACEMENTS:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    return text

def add_calm_insertion(text: str, probability: float = 0.5) -> str:
    """Добавляет успокаивающие вставки"""
    if random.random() < probability:
        insertion = random.choice(CALM_INSERTIONS)
        sentences = text.split('. ')
        if len(sentences) > 2:
            insert_pos = random.randint(1, len(sentences)-1)
            sentences[insert_pos] = sentences[insert_pos] + insertion
            return '. '.join(sentences)
        else:
            return text + insertion
    return text

def add_reassuring_start(text: str, probability: float = 0.3) -> str:
    """Добавляет успокаивающее начало"""
    if random.random() < probability and len(text) > 50:
        start = random.choice(REASSURING_STARTS)
        if text[0].isupper():
            text = text[0].lower() + text[1:]
        return start + text
    return text

def add_peaceful_ending(text: str, probability: float = 0.4) -> str:
    """Добавляет мирную концовку"""
    if random.random() < probability:
        ending = random.choice(PEACEFUL_ENDINGS)
        if text.endswith('.'):
            text = text[:-1]
        return text + ending
    return text

def soften_questions(text: str) -> str:
    """Смягчает тревожные вопросы"""
    anxious_questions = [
        (r'\bчто если (?!всё наладится)', 'допустим, даже если'),
        (r'\bа вдруг', 'даже если предположить, что'),
        (r'\bпочему я', 'ну, возможно, я'),
    ]
    for pattern, repl in anxious_questions:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    
    for pattern, repl in SOFTENING_QUESTIONS:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    
    return text

def add_mindfulness_phrase(text: str, probability: float = 0.35) -> str:
    """Добавляет осознанные, медитативные фразы"""
    mindfulness = [
        " Я просто наблюдаю за мыслями, не цепляясь за них.",
        " Я здесь и сейчас, и в этот момент всё хорошо.",
        " Я дышу спокойно и чувствую, как напряжение уходит.",
        " Это просто мысль. Мысль не равна реальности.",
        " Я разрешаю себе быть спокойным без причины.",
    ]
    if random.random() < probability:
        return text + random.choice(mindfulness)
    return text

def reduce_catastrophization(text: str) -> str:
    """Убирает катастрофические обороты"""
    catastrophes = [
        (r'схожу с ума', 'немного устал'),
        (r'это конец', 'просто изменения'),
        (r'всё рушится', 'всё меняется постепенно'),
        (r'никогда не справлюсь', 'справлюсь со временем'),
        (r'ужас что будет', 'посмотрим, что будет'),
    ]
    for pattern, repl in catastrophes:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    return text

def calm_text(text: str, intensity: float = 0.7) -> str:
    """
    Главная функция усмирения текста (делает его нетревожным)
    intensity: 0 (минимальное) - 1 (максимальное успокоение)
    """
    original = text
    
    # 1. Замена тревожных слов на спокойные
    text = replace_with_calm(text)
    
    # 2. Убираем усилители
    text = remove_intensifiers(text)
    
    # 3. Убираем катастрофизацию
    text = reduce_catastrophization(text)
    
    # 4. Смягчаем вопросы
    text = soften_questions(text)
    
    # 5. Добавляем успокаивающее начало 
    if random.random() < intensity * 0.4:
        text = add_reassuring_start(text, probability=0.4)
    
    # 6. Успокаивающие вставки
    if random.random() < intensity * 0.5:
        text = add_calm_insertion(text, probability=0.5)
    
    # 7. Осознанные фразы
    if random.random() < intensity * 0.4:
        text = add_mindfulness_phrase(text, probability=0.5)
    
    # 8. Мирная концовка
    if random.random() < intensity * 0.5:
        text = add_peaceful_ending(text, probability=0.6)
    
    # Защита от пустого текста
    if len(text.strip()) < len(original.strip()) * 0.2:
        text = original + " В целом, я спокоен и принимаю ситуацию."
    
    return text

def process_dataframe(
    df_positive: pd.DataFrame,
    df_negative: pd.DataFrame,
    calm_intensity: float = 0.7,
    text_column: str = 'text'
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Тревожные -> усиливаем тревожность (из прошлого скрипта)
    Нетревожные -> сильно успокаиваем
    """
    df_negative_calmed = df_negative.copy()
    
    print("Успокаиваю нетревожные тексты...")
    df_negative_calmed[text_column] = df_negative_calmed[text_column].apply(
        lambda x: calm_text(str(x), intensity=calm_intensity)
    )
    
    df_positive_unchanged = df_positive.copy()
    
    return df_positive_unchanged, df_negative_calmed

# ========== ОСНОВНАЯ ЧАСТЬ ==========

if __name__ == "__main__":
    random.seed(42)
    
    print("="*70)
    print("УСПОКОИТЕЛЬ ТРЕВОЖНЫХ ТЕКСТОВ")
    print("="*70)
    
    try:
        df_positive = pd.read_csv('anxiety_train_x_positive.csv')
        df_negative = pd.read_csv('anxiety_train_x_negative.csv')
    except FileNotFoundError as e:
        print(f"Ошибка: {e}")
        print("Файлы anxiety_test_x_positive.csv и anxiety_test_x_negative.csv должны быть в папке")
        exit()
    
    print(f"\n📊 Исходные данные:")
    print(f"   Тревожных текстов: {len(df_positive)}")
    print(f"   Нетревожных текстов: {len(df_negative)}")
    
    # Интенсивность успокоения (0.5 - лёгкая, 0.7 - средняя, 0.9 - сильная)
    CALM_INTENSITY = 0.75
    
    print(f"\n Настройки:")
    print(f"   Интенсивность успокоения: {CALM_INTENSITY}")
    print(f"   Режим: мягкое принятие + осознанность")
    
    print("\nПрименяю успокоение к нетревожным текстам...")
    df_pos_unchanged, df_neg_calmed = process_dataframe(
        df_positive, df_negative,
        calm_intensity=CALM_INTENSITY,
        text_column='text'
    )
    
    OUTPUT_POSITIVE = 'anxiety_train_x_positive_unchanged.csv'
    OUTPUT_NEGATIVE = 'anxiety_train_x_negative_calmed.csv'
    
    df_pos_unchanged.to_csv(OUTPUT_POSITIVE, index=False)
    df_neg_calmed.to_csv(OUTPUT_NEGATIVE, index=False)
    
    print(f"\nГОТОВО!")
    print(f"   Тревожные тексты (оригиналы): {OUTPUT_POSITIVE}")
    print(f"   Успокоенные нетревожные тексты: {OUTPUT_NEGATIVE}")
    
    print("\n" + "="*70)
    print("ПРИМЕРЫ УСПОКОЕНИЯ:")
    print("="*70)
    
    for i in range(min(3, len(df_negative))):
        print(f"\n--- Пример {i+1} (нетревожный текст) ---")
        original = df_negative.iloc[i]['text']
        calmed = df_neg_calmed.iloc[i]['text']
        
        print("ОРИГИНАЛ (первые 200 символов):")
        print(original[:200] + "..." if len(original) > 200 else original)
        
        print("\nУСПОКОЕННЫЙ (первые 350 символов):")
        print(calmed[:350] + "..." if len(calmed) > 350 else calmed)
        print("-"*70)
    
    print("\nСТАТИСТИКА ИЗМЕНЕНИЙ:")
    orig_lens = df_negative['text'].str.len().mean()
    calm_lens = df_neg_calmed['text'].str.len().mean()
    print(f"   Средняя длина оригиналов: {orig_lens:.0f} символов")
    print(f"   Средняя длина успокоенных: {calm_lens:.0f} символов")
    
    print("\nСовет: теперь ваша модель будет видеть чёткую границу")
    print("   — тревожные тексты: напряжение, соматика, катастрофизация")
    print("   — нетревожные тексты: принятие, спокойствие, размеренность")

УСПОКОИТЕЛЬ ТРЕВОЖНЫХ ТЕКСТОВ

📊 Исходные данные:
   Тревожных текстов: 266
   Нетревожных текстов: 227

⚙️  Настройки:
   Интенсивность успокоения: 0.75
   Режим: мягкое принятие + осознанность

🔄 Применяю успокоение к нетревожным текстам...
Успокаиваю нетревожные тексты...

✅ ГОТОВО!
   📁 Тревожные тексты (оригиналы): anxiety_train_x_positive_unchanged.csv
   📁 Успокоенные нетревожные тексты: anxiety_train_x_negative_calmed.csv

📝 ПРИМЕРЫ УСПОКОЕНИЯ:

--- Пример 1 (нетревожный текст) ---
ОРИГИНАЛ (первые 200 символов):
Здорово, дружище! Давно не виделись. У меня все нормально. Продолжаю потихоньку копить ролевой опыт. Перебираемся с третьей дынды на пятерку, и черт возьми, как же легко там генериться. В трешке на од...

УСПОКОЕННЫЙ (первые 350 символов):
Здорово, дружище! Давно не виделись. У меня все нормально. Продолжаю потихоньку копить ролевой опыт. Перебираемся с третьей дынды на пятерку, и черт возьми, как же легко там генериться. В трешке на одно только распределение скилл-пой

In [10]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle

# ========== КОНФИГУРАЦИЯ ==========
POSITIVE_FILE = 'anxiety_train_x_positive_amplified.csv'  
NEGATIVE_FILE = 'anxiety_train_x_negative_calmed.csv'     

OUTPUT_X = 'anxiety_train_x_amplified.csv'  
OUTPUT_Y = 'anxiety_train_y_amplified.csv'   

LABEL_POSITIVE = 'ТРЕВОЖНОСТЬ'
LABEL_NEGATIVE = 'НЕТ ТРЕВОЖНОСТИ'

# ========== ЗАГРУЗКА ДАННЫХ ==========
print("="*70)
print("ОБЪЕДИНЕНИЕ И ПЕРЕМЕШИВАНИЕ ДАННЫХ")
print("="*70)

try:
    df_positive = pd.read_csv(POSITIVE_FILE)
    df_negative = pd.read_csv(NEGATIVE_FILE)
    print(f"\nЗагружены файлы:")
    print(f"   Тревожные тексты: {len(df_positive)} записей")
    print(f"   Нетревожные тексты: {len(df_negative)} записей")
except FileNotFoundError as e:
    print(f"\nОшибка: {e}")
    print("Убедитесь, что файлы существуют в текущей папке")
    exit()

# ========== ДОБАВЛЕНИЕ МЕТОК ==========
df_positive['label'] = LABEL_POSITIVE
df_negative['label'] = LABEL_NEGATIVE

# ========== ОБЪЕДИНЕНИЕ ==========
df_combined = pd.concat([df_positive, df_negative], ignore_index=True)

print(f"\nОбъединённый датасет:")
print(f"   Всего записей: {len(df_combined)}")
print(f"   Тревожных: {len(df_positive)} ({len(df_positive)/len(df_combined)*100:.1f}%)")
print(f"   Нетревожных: {len(df_negative)} ({len(df_negative)/len(df_combined)*100:.1f}%)")

# ========== ПЕРЕМЕШИВАНИЕ ==========
df_shuffled = shuffle(df_combined, random_state=42).reset_index(drop=True)

print(f"\nДанные перемешаны (random_state=42)")

# ========== РАЗДЕЛЕНИЕ НА X И Y ==========
text_column = 'text'
if text_column not in df_shuffled.columns:
    print(f"\n⚠️  Колонка '{text_column}' не найдена. Доступные колонки: {df_shuffled.columns.tolist()}")
    possible_text_cols = ['text', 'Text', 'TEXT', 'content', 'message', 'sentence']
    for col in possible_text_cols:
        if col in df_shuffled.columns:
            text_column = col
            print(f"   Использую колонку: '{text_column}'")
            break
    else:
        print("   Не найдена текстовая колонка. Использую первую колонку")
        text_column = df_shuffled.columns[0]

df_x = df_shuffled[[text_column]].copy()
df_x.columns = ['text'] 

df_y = df_shuffled[['label']].copy()

# ========== СОХРАНЕНИЕ ==========
df_x.to_csv(OUTPUT_X, index=False)
df_y.to_csv(OUTPUT_Y, index=False)

print(f"\nФайлы сохранены:")
print(f"   {OUTPUT_X} — только тексты ({len(df_x)} записей)")
print(f"   {OUTPUT_Y} — только метки ({len(df_y)} записей)")

# ========== ПРОВЕРКА СООТВЕТСТВИЯ ==========
print("\n" + "="*70)
print("ПРОВЕРКА СООТВЕТСТВИЯ X И Y")
print("="*70)

assert len(df_x) == len(df_y), "ОШИБКА: количество строк в X и Y не совпадает!"

print(f"\nРаспределение меток в {OUTPUT_Y}:")
label_counts = df_y['label'].value_counts()
for label, count in label_counts.items():
    print(f"   {label}: {count} ({count/len(df_y)*100:.1f}%)")

# ========== ПРИМЕРЫ ПЕРВЫХ ЗАПИСЕЙ ==========
print("\n" + "="*70)
print("ПРИМЕРЫ ПЕРВЫХ 5 ЗАПИСЕЙ (ПОСЛЕ ПЕРЕМЕШИВАНИЯ)")
print("="*70)

for i in range(min(5, len(df_x))):
    print(f"\n--- Запись {i+1} ---")
    print(f"Метка: {df_y.iloc[i]['label']}")
    text_preview = df_x.iloc[i]['text']
    if len(text_preview) > 200:
        text_preview = text_preview[:200] + "..."
    print(f"Текст: {text_preview}")

# ========== ВАЛИДАЦИЯ ==========
print("\n" + "="*70)
print("ВАЛИДАЦИЯ")
print("="*70)

null_x = df_x['text'].isnull().sum()
null_y = df_y['label'].isnull().sum()

print(f"Пропуски в X: {null_x}")
print(f"Пропуски в Y: {null_y}")

unique_labels = df_y['label'].unique()
print(f"Уникальные метки: {unique_labels}")

if len(unique_labels) != 2:
    print(f"ВНИМАНИЕ: ожидалось 2 класса, найдено {len(unique_labels)}")

print(f"Порядок записей: X и Y синхронизированы (оба имеют {len(df_x)} строк)")

print("\n" + "="*70)
print("ГОТОВО! Можно использовать файлы для обучения/тестирования модели")
print("="*70)

# ========== ОПЦИОНАЛЬНО: СОЗДАНИЕ ТРЕТЬЕГО ФАЙЛА С ОБЕИМИ КОЛОНКАМИ ==========
CREATE_FULL_FILE = True  

if CREATE_FULL_FILE:
    OUTPUT_FULL = 'anxiety_train_amplified_full.csv'
    df_full = pd.concat([df_x, df_y], axis=1)
    df_full.to_csv(OUTPUT_FULL, index=False)
    print(f"\nДополнительно создан файл с обеими колонками: {OUTPUT_FULL}")

ОБЪЕДИНЕНИЕ И ПЕРЕМЕШИВАНИЕ ДАННЫХ

✅ Загружены файлы:
   Тревожные тексты: 266 записей
   Нетревожные тексты: 227 записей

📊 Объединённый датасет:
   Всего записей: 493
   Тревожных: 266 (54.0%)
   Нетревожных: 227 (46.0%)

🔄 Данные перемешаны (random_state=42)

✅ Файлы сохранены:
   📁 anxiety_train_x_amplified.csv — только тексты (493 записей)
   📁 anxiety_train_y_amplified.csv — только метки (493 записей)

ПРОВЕРКА СООТВЕТСТВИЯ X И Y

📊 Распределение меток в anxiety_train_y_amplified.csv:
   ТРЕВОЖНОСТЬ: 266 (54.0%)
   НЕТ ТРЕВОЖНОСТИ: 227 (46.0%)

ПРИМЕРЫ ПЕРВЫХ 5 ЗАПИСЕЙ (ПОСЛЕ ПЕРЕМЕШИВАНИЯ)

--- Запись 1 ---
Метка: НЕТ ТРЕВОЖНОСТИ
Текст: Сложно оценить ситуацию. Не думаю, что это скоро продет. Я отпускаю тревогу и просто наблюдаю за ситуацией.

--- Запись 2 ---
Метка: ТРЕВОЖНОСТЬ
Текст: Привет, мой друг далекий, как долго мы не виделись с тобой! И вот пишу тебе письмо, как твои дела? Что у тебя изменилось с тех пор? Я так скучаю  У меня все хорошо, учусь. уже совсем скоро, и это